Transformer 模型由于其**自注意力机制（Self-Attention）**是对输入序列进行并行处理的，天然无法捕捉 Token 之间的顺序关系。为了引入位置信息，Transformer 采用了**位置编码（Positional Encoding）**。

以下是关于位置编码原理与公式的整理笔记：

---

## 1. 核心设计哲学
位置编码需要满足几个关键条件：
* **独特性**：每个位置的编码应该是唯一的。
* **确定性**：对于不同长度的序列，相同位置的编码偏移量应保持一致。
* **泛化性**：能够处理比训练时更长的序列。
* **线性关系**：位置编码应包含相对位置信息，使模型能容易地学习到位置间的线性关系。

---

## 2. 正弦/余弦位置编码公式
Transformer 原作（*Attention is All You Need*）中采用了基于不同频率的正弦和余弦函数的绝对位置编码。

### 公式定义
对于序列中的位置 $pos$ 和维度索引 $i$，$d_{model}$ 为编码的总维度，位置编码 $PE$ 的计算如下：

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

* **$pos$**：Token 在序列中的绝对位置（0, 1, 2, ...）。
* **$i$**：维度索引，取值范围是 $[0, d_{model}/2)$。
* **$10000$**：一个人为设定的衰减常数，用于控制波长的范围。



---

## 3. 原理解析：为什么是波函数？

### A. 相对位置的可学习性
选择正余弦函数的一个核心原因是：对于任何固定的偏移量 $k$，$PE_{pos+k}$ 可以表示为 $PE_{pos}$ 的线性函数。
根据三角函数展开公式：
$$\sin(A+B) = \sin A \cos B + \cos A \sin B$$
$$\cos(A+B) = \cos A \cos B - \sin A \sin B$$
这意味着模型可以通过注意力机制中的线性变换，轻松捕捉到 Token 之间的**相对距离**。

### B. 不同维度的频率变化
* **低维度（$i$ 较小）**：波长较短，频率较高，负责捕捉近距离的精细位置变化。
* **高维度（$i$ 较大）**：波长较长，频率较低，负责捕捉长距离的宏观位置信息。
这类似于二进制编码：低位变动快，高位变动慢。

---

## 4. 编码的注入方式
在 Transformer 中，位置编码**不是拼接（Concatenate）**，而是直接与词嵌入（Word Embedding）**相加（Element-wise Add）**：

$$X_{final} = Embedding(X) + PE$$

> **直觉理解**：由于词嵌入通常位于高维空间的子空间中，相加操作相当于给语义向量穿上了一件带有“刻度”的外衣，而不会破坏原本的语义信息。

---

## 5. 其他位置编码方案（对比）

| 类型 | 描述 | 代表模型 |
| :--- | :--- | :--- |
| **绝对位置嵌入** | 将位置视为 ID，学习一个可训练的 Embedding 层。 | BERT, GPT-2 |
| **旋转位置编码 (RoPE)** | 通过旋转矩阵将相对位置信息注入 Key 和 Query 的点积中。 | Llama, PaLM |
| **相对位置偏差 (ALiBi)** | 根据距离在 Attention Score 上直接加上一个负偏置。 | Bloom, MPT |

---

## 6. 总结提炼
* **本质**：用一组周期性函数在特征维度上构造唯一的指纹。
* **优点**：无需额外参数，支持无限长度的外推（虽然实际效果有限），保留了相对位置的线性性质。
* **应用**：在当前主流的大模型中，虽然原始的 Sinusoidal 编码仍被提及，但 **RoPE (Rotary Positional Embedding)** 已经成为事实上的工业标准。

In [1]:
import torch
import torch.nn as nn
import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # (1, max_len, d_model) 方便广播
        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        output = x + self.pe[:, : x.size(1)]
        print("\n=== 位置编码输出 ===")
        print(f"形状: {output.shape}")
        print("位置编码后的第一个token:", output[0, 0, :3].detach().numpy())
        return output


In [3]:
# ===================== 测试代码 =====================
def test_positional_encoding():
    torch.manual_seed(42)

    batch_size = 2
    seq_len = 5
    d_model = 6

    # 模拟输入 (全0，方便观察位置编码本身)
    x = torch.zeros(batch_size, seq_len, d_model)

    print("=== 输入 ===")
    print(f"形状: {x.shape}")
    print("第一个token:", x[0, 0])

    pe = PositionalEncoding(d_model)

    output = pe(x)

    print("\n=== 验证 ===")

    # 1. shape 检查
    print("shape是否一致:", output.shape == x.shape)

    # 2. 不同位置是否不同
    print("pos0 vs pos1 是否不同:", not torch.allclose(output[0, 0], output[0, 1]))

    # 3. 同一位置在不同 batch 是否一样
    print(
        "batch0 pos0 vs batch1 pos0 是否一样:",
        torch.allclose(output[0, 0], output[1, 0]),
    )

    # 4. 打印前两个位置编码
    print("\n位置0:", output[0, 0])
    print("位置1:", output[0, 1])


test_positional_encoding()

=== 输入 ===
形状: torch.Size([2, 5, 6])
第一个token: tensor([0., 0., 0., 0., 0., 0.])

=== 位置编码输出 ===
形状: torch.Size([2, 5, 6])
位置编码后的第一个token: [0. 1. 0.]

=== 验证 ===
shape是否一致: True
pos0 vs pos1 是否不同: True
batch0 pos0 vs batch1 pos0 是否一样: True

位置0: tensor([0., 1., 0., 1., 0., 1.])
位置1: tensor([0.8415, 0.5403, 0.0464, 0.9989, 0.0022, 1.0000])
